# Tests for blend_scores.py
This notebook provides basic tests for the blending logic and scoring functions in `blend_scores.py`.

In [14]:
import sys
import os
from dotenv import load_dotenv

# === 1. Setup Project Root and Environment ===
# Get the project root (SilverKey directory)
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "../../../.."))

# === 2. Add Server directory to Python path ===
server_dir = os.path.join(project_root, "Server")
if server_dir not in sys.path:
    sys.path.insert(0, server_dir)
print(f"📁 Added to sys.path: {server_dir}")

# === 3. Import components using absolute imports ===
try:
    from app.home_matching.ensemble.blend_scores import EnsembleScorer, blend_scores as blend_scores_func, score_user_home_pair
    print("✅ All LLM scorer components imported successfully!")
except Exception as e:
    print(f"❌ Failed to import components: {e}")

import numpy as np

📁 Added to sys.path: /Users/jaycewalzer/Desktop/SilverKey/Server
✅ All LLM scorer components imported successfully!


## Test blend_scores function (standalone)

In [15]:
# Test with equal weights and mid-range scores
score = blend_scores_func(0.5, 0.5, 0.5)
print('Blended score (0.5, 0.5, 0.5):', score)
assert 0.0 <= score <= 1.0


/Users/jaycewalzer/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Blended score (0.5, 0.5, 0.5): 0.5


## Test EnsembleScorer class with dummy data (mocking model outputs)

In [16]:
class DummyScorer:
    def score_user_against_homes(self, user_data, homes_data):
        # Return home_id and a dummy score for each home
        return [(home['home_id'], 0.7) for home in homes_data]
    def batch_score_homes(self, user_data, homes_data):
        return [0.6 for _ in homes_data]
    def llm_score(self, user_data, home_data):
        return 0.8
    def predict_batch(self, user_data, homes_data):
        return [0.9 for _ in homes_data]


In [17]:
# Patch EnsembleScorer to use dummy scorers
EnsembleScorer.embedding_scorer = DummyScorer()
EnsembleScorer.tabular_predictor = DummyScorer()
EnsembleScorer.llm_scorer = DummyScorer()
scorer = EnsembleScorer(embedding_weight=0.3, tabular_weight=0.3, llm_weight=0.4)
user_data = {'user_id': 'u1'}
home_data = {'home_id': 'h1'}
result = scorer.score_user_home_pair(user_data, home_data)
print('Score user-home pair:', result)
assert 'final_score' in result


Score user-home pair: {'user_id': 'u1', 'home_id': 'h1', 'scores': {'embedding': 0.26801273112151486, 'tabular': 0.5954710245132446, 'llm': 0.0}, 'final_score': 0.25904512669042784, 'method_weights': {'embedding': 0.3, 'tabular': 0.3, 'llm': 0.4}}


## Test score_user_home_pair convenience function

In [18]:
# This will use real models unless patched
try:
    result = score_user_home_pair({'user_id': 'u2'}, {'home_id': 'h2'})
    print('Convenience function result:', result)
except Exception as e:
    print('Expected error (no model files):', e)


Convenience function result: {'user_id': 'u2', 'home_id': 'h2', 'scores': {'embedding': 0.26801273112151486, 'tabular': 0.5954710245132446, 'llm': 0.0}, 'final_score': 0.3453935022539038, 'method_weights': {'embedding': 0.4, 'tabular': 0.4, 'llm': 0.2}}
